In [ ]:
import getpass
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

clave_pg = getpass.getpass("Contraseña de PostgreSQL: ")

url = URL.create(
    "postgresql+psycopg2",
    username="postgres",
    password=clave_pg,
    host="localhost",
    port=5432,
    database="tesis_agronomia",
)
engine = create_engine(url)

In [ ]:
# Nombres reales de las sedes: config_local.py (no se publica; ver config_local.ejemplo.py)
from config_local import SEDE_CENTRAL, SEDE_INTERIOR

In [ ]:
import pandas as pd

df = pd.read_sql("SELECT * FROM tesis.pbi_estudiantes", engine)
print(df.columns.tolist())   # para ver los nombres de las columnas

# Solo los 985 que completaron el primer año (tienen tiempo)
m = df[df["tiempo"].notna()].copy()
m["evento"] = m["evento"].astype(int)

def km(datos, t_final=4):
    s = 1.0
    for t in range(0, t_final + 1):
        en_riesgo = (datos["tiempo"] >= t).sum()
        desertan = ((datos["tiempo"] == t) & (datos["evento"] == 1)).sum()
        if en_riesgo > 0:
            s *= 1 - desertan / en_riesgo
    return round(100 * s, 1)

print("Control total:", km(m), "(debe dar 67.0)")
print(m.groupby("cohorte").apply(km))

In [ ]:
# --- Bibliotecas y datos de supervivencia (985 estudiantes) ---
import numpy as np
import matplotlib.pyplot as plt
from sksurv.nonparametric import kaplan_meier_estimator

tr = pd.read_sql("SELECT * FROM tesis.trayectorias WHERE grupo = 'modelo'", engine)

print("Estudiantes:", len(tr), "| desertores:", int(tr["evento"].sum()))
print("Columnas:", list(tr.columns))

In [ ]:
# --- Curva general corregida: empieza en 1 en el tiempo 0 ---
from matplotlib.ticker import FuncFormatter

t, s, (ic_inf, ic_sup) = kaplan_meier_estimator(
    tr["evento"].astype(bool), tr["tiempo"], conf_type="log-log")
t, s = np.r_[0, t], np.r_[1, s]
ic_inf, ic_sup = np.r_[1, ic_inf], np.r_[1, ic_sup]

def en_curva(x):                      # valor de la curva escalonada en x
    return s[np.searchsorted(t, x, side="right") - 1]

print("Control: S(2) =", round(en_curva(2), 3), " S(4) =", round(en_curva(4), 3),
      " S(8) =", round(en_curva(8), 3))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.step(t, s, where="post", color="black", linewidth=2, label="Permanencia estimada")
ax.fill_between(t, ic_inf, ic_sup, step="post", alpha=0.2, color="gray", label="IC 95 %")
cens = tr.loc[tr["evento"] == 0, "tiempo"]
ax.plot(cens, en_curva(cens.values), "|", color="0.4", markersize=8, label="Censurados")

ax.set_xlabel("Cuatrimestres desde el final del primer año")
ax.set_ylabel("Proporción que permanece")
ax.set_ylim(0, 1.02)
ax.set_xticks(range(0, 21, 2))
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:.1f}".replace(".", ",")))
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig("Figura_4_3_km_general.png", dpi=300)
plt.show()

In [ ]:
from sksurv.compare import compare_survival
from matplotlib.ticker import FuncFormatter

def curva(df):
    t, s = kaplan_meier_estimator(df["evento"].astype(bool), df["tiempo"])
    return np.r_[0, t], np.r_[1, s]          # la curva empieza en 1 en el tiempo 0

def valor_en(t, s, x):
    return s[t <= x][-1]

# --- Datos: los 978 con perfil ---
gr = pd.read_sql("SELECT id_est, grupo_km, perfil FROM tesis.grupos", engine)
d = tr.merge(gr, on="id_est")
print("Estudiantes:", len(d), "\n")

# --- Estilo de cada perfil (grises + tipo de línea) ---
estilo = {
    "Participación baja": {"color": "0.0",  "linestyle": "-"},
    "Cursada parcial":    {"color": "0.35", "linestyle": "--"},
    "Cursada sostenida":  {"color": "0.6",  "linestyle": ":"},
}

# --- Resumen por perfil ---
filas = []
fig, ax = plt.subplots(figsize=(8, 4.5))
for nombre in estilo:                         # orden fijo de la leyenda
    sub = d[d["perfil"] == nombre]
    t, s = curva(sub)
    ax.step(t, s, where="post", label=nombre, linewidth=2, **estilo[nombre])
    med = t[s <= 0.5].min() if (s <= 0.5).any() else None
    filas.append({"perfil": nombre, "estudiantes": len(sub),
                  "desertores": int(sub["evento"].sum()),
                  "S(2)": valor_en(t, s, 2), "S(4)": valor_en(t, s, 4),
                  "S(8)": valor_en(t, s, 8), "mediana": med})
print(pd.DataFrame(filas).round(3).to_string(index=False), "\n")

# --- Prueba de log-rank: ¿las curvas difieren? ---
y = np.array(list(zip(d["evento"].astype(bool), d["tiempo"])),
             dtype=[("evento", bool), ("tiempo", float)])
chi2, p = compare_survival(y, d["perfil"].values)
print(f"Log-rank: chi² = {chi2:.1f}, p = {p:.4g}")

# --- Formato del gráfico ---
ax.set_xlabel("Cuatrimestres desde el final del primer año")
ax.set_ylabel("Proporción que permanece")
ax.set_ylim(0, 1.02)
ax.set_xticks(range(0, 21, 2))
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:.1f}".replace(".", ",")))
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig("Figura_4_4_km_perfiles.png", dpi=300)
plt.show()

In [ ]:
# --- Kaplan-Meier por sede (985) ---
from matplotlib.ticker import FuncFormatter

# Etiquetas anonimizadas: solo cambian los nombres que se muestran
ETIQ_SEDE = {SEDE_INTERIOR: "Sede del interior",
             SEDE_CENTRAL: "Sede Central"}
print("Valores de sede en los datos:", tr["sede_ingreso"].unique())
faltan = set(tr["sede_ingreso"].unique()) - set(ETIQ_SEDE)
assert not faltan, f"Falta etiqueta para: {faltan}"

# Estilo de cada sede (grises + tipo de línea), en orden fijo
estilo = {"Sede Central":      {"color": "0.5", "linestyle": "--"},
          "Sede del interior": {"color": "0.0", "linestyle": "-"}}

filas, riesgo = [], {}
fig, ax = plt.subplots(figsize=(8, 4.5))
for nombre in estilo:
    sub = tr[tr["sede_ingreso"].map(ETIQ_SEDE) == nombre]
    t, s = curva(sub)
    ax.step(t, s, where="post", label=nombre, linewidth=2, **estilo[nombre])
    med = t[s <= 0.5].min() if (s <= 0.5).any() else None
    filas.append({"sede": nombre, "estudiantes": len(sub), "desertores": int(sub["evento"].sum()),
                  "S(2)": valor_en(t, s, 2), "S(4)": valor_en(t, s, 4),
                  "S(6)": valor_en(t, s, 6), "S(8)": valor_en(t, s, 8), "mediana": med})
    # riesgo por cuatrimestre = desertores en t / estudiantes en riesgo en t
    riesgo[nombre] = {k: sub[(sub["tiempo"] == k) & (sub["evento"] == 1)].shape[0] / (sub["tiempo"] >= k).sum()
                      for k in range(1, 11)}
print(pd.DataFrame(filas).round(3).to_string(index=False), "\n")

print("Riesgo de desertar en cada cuatrimestre (tiempos 1 a 10):")
print(pd.DataFrame(riesgo).round(3).T.to_string(), "\n")

y_all = np.array(list(zip(tr["evento"].astype(bool), tr["tiempo"])),
                 dtype=[("evento", bool), ("tiempo", float)])
chi2, p = compare_survival(y_all, tr["sede_ingreso"].values)
print(f"Log-rank: chi² = {chi2:.1f}, p = {p:.4g}")

# --- Formato del gráfico ---
ax.axvline(4, color="0.7", linestyle=":", linewidth=1)
ax.set_xlabel("Cuatrimestres desde el final del primer año")
ax.set_ylabel("Proporción que permanece")
ax.set_ylim(0, 1.02)
ax.set_xticks(range(0, 21, 2))
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:.1f}".replace(".", ",")))
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig("Figura_4_5_km_sedes.png", dpi=300)
plt.show()

In [ ]:
# --- Tiempo calendario de cada estudiante ---
tr["inicio"] = (tr["cohorte"] - 2015) * 2          # 1.er cuatrimestre de la cohorte
tr["arranque"] = tr["inicio"] + 1                  # fin del primer año = tiempo 0
tr["fin_cal"] = tr["arranque"] + tr["tiempo"]      # cuatrimestre calendario del evento o la censura

# --- Permanencia por cohorte ---
filas = []
for c, sub in tr.groupby("cohorte"):
    t, s = curva(sub)
    filas.append({"cohorte": c, "estudiantes": len(sub), "desertores": int(sub["evento"].sum()),
                  "S(2)": valor_en(t, s, 2),
                  "S(4)": valor_en(t, s, 4) if sub["tiempo"].max() >= 4 else None})
print(pd.DataFrame(filas).round(3).to_string(index=False), "\n")

chi2, p = compare_survival(y_all, tr["cohorte"].values)
print(f"Log-rank entre cohortes: chi² = {chi2:.1f}, p = {p:.4g}\n")

# --- Riesgo por año calendario (todas las cohortes juntas) ---
filas = []
for cal in range(1, 22):
    en_riesgo = ((tr["arranque"] < cal) & (tr["fin_cal"] >= cal)).sum()
    eventos = ((tr["evento"] == 1) & (tr["fin_cal"] == cal)).sum()
    filas.append({"anio": 2015 + cal // 2, "en_riesgo": en_riesgo, "desertores": eventos})
cal = pd.DataFrame(filas).groupby("anio").sum()
cal["riesgo"] = cal["desertores"] / cal["en_riesgo"]
print("Riesgo por año calendario (desertores / estudiante-cuatrimestres en riesgo):")
print(cal.round(3).to_string())

In [ ]:
est = pd.read_sql("SELECT id_est, cohorte, sede_ingreso FROM tesis.estudiantes", engine)
tra = pd.read_sql("SELECT id_est, inicio, ultima, grupo FROM tesis.trayectorias", engine)
cur = pd.read_sql("SELECT id_est, cuatri, estado FROM tesis.cursadas", engine)

base = est.merge(tra, on="id_est").set_index("id_est")

# --- Cursadas del 1.er cuatrimestre de cada estudiante ---
c1 = cur.merge(base[["inicio"]].reset_index(), on="id_est")
c1 = c1[c1["cuatri"] == c1["inicio"]]
base = base.join(c1.groupby("id_est").size().rename("cursadas_1c"))
base = base.join(c1.assign(a=c1["estado"].eq("ausente")).groupby("id_est")["a"].mean().rename("prop_aus_1c"))
base["cursadas_1c"] = base["cursadas_1c"].fillna(0)

# 1) ¿Todos tienen la misma cantidad de cursadas en el 1.er cuatrimestre? (por cohorte y sede)
print(base.groupby(["cohorte", "sede_ingreso"])["cursadas_1c"]
          .describe()[["count", "mean", "50%", "min", "max", "std"]].round(1).to_string(), "\n")

# 2) Posibles «no iniciados»: actividad solo en el 1.er cuatrimestre y todas las cursadas ausentes
ni = (base["ultima"] == base["inicio"]) & (base["prop_aus_1c"].eq(1) | base["prop_aus_1c"].isna())
print("Posibles no iniciados:", ni.sum(), "de", len(base), "estudiantes")
print(base[ni].groupby("cohorte").size().to_string())

In [ ]:
import pandas as pd
from sqlalchemy import text

consulta = text("""
    SELECT i.id_est, i.cohorte, i.sede_ingreso,
           i.materias_cursadas, i.prop_ausentes, i.tasa_regularizacion,
           i.materias_aprobadas, i.ingreso_equiv,
           t.tiempo, t.evento
    FROM tesis.indicadores i
    JOIN tesis.trayectorias t ON t.id_est = i.id_est
    WHERE i.incluido
""")
with engine.connect() as con:
    dm = pd.read_sql(consulta, con)

print("Filas:", len(dm))
print("\nNulos por columna:\n", dm.isna().sum())
print("\nEvento (1 = desertó):\n", dm["evento"].value_counts())
print("\nSede x evento:\n", pd.crosstab(dm["sede_ingreso"], dm["evento"], margins=True))
print("\nIngreso por equivalencias:", int(dm["ingreso_equiv"].sum()))
print("\nTiempo: mín", dm["tiempo"].min(), "– máx", dm["tiempo"].max())
print("\nTipos:\n", dm.dtypes)

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

X = dm[["materias_cursadas", "prop_ausentes", "tasa_regularizacion", "materias_aprobadas"]].copy()
X["sede_sp"] = (dm["sede_ingreso"] == SEDE_INTERIOR).astype(int)
X["cohorte"] = dm["cohorte"]
X["ingreso_equiv"] = dm["ingreso_equiv"].astype(int)

def vif(X):
    res = {}
    for c in X.columns:
        otras = X.drop(columns=c)
        r2 = LinearRegression().fit(otras, X[c]).score(otras, X[c])
        res[c] = 1 / (1 - r2)
    return pd.Series(res).round(2)

print("Con tasa de regularización:\n", vif(X))
print("\nSin tasa de regularización:\n", vif(X.drop(columns="tasa_regularizacion")))

In [ ]:
from sksurv.util import Surv

periodo = pd.cut(dm["cohorte"], bins=[2014, 2018, 2021, 2023],
                 labels=["2015-2018", "2019-2021", "2022-2023"])

Xm = pd.DataFrame({
    "materias_cursadas":  dm["materias_cursadas"],
    "ausentes_x10":       dm["prop_ausentes"] * 10,
    "materias_aprobadas": dm["materias_aprobadas"],
    "sede_sp":            (dm["sede_ingreso"] == SEDE_INTERIOR).astype(int),
    "coh_2019_2021":      (periodo == "2019-2021").astype(int),
    "coh_2022_2023":      (periodo == "2022-2023").astype(int),
    "ingreso_equiv":      dm["ingreso_equiv"].astype(int),
})
ym = Surv.from_arrays(event=dm["evento"].astype(bool), time=dm["tiempo"])

print("Xm:", Xm.shape)
print(Xm.describe().T[["mean", "min", "max"]].round(2))
print("\nPeríodo x evento:\n", pd.crosstab(periodo, dm["evento"], margins=True))
print("\nym:", ym.dtype, "| eventos:", ym["event"].sum())
print("Primeros 3:", ym[:3])

In [ ]:
from sklearn.model_selection import train_test_split

estrato = dm["evento"].astype(int).astype(str) + "_" + dm["sede_ingreso"]
idx_ent, idx_pru = train_test_split(dm.index, test_size=0.25,
                                    stratify=estrato, random_state=2026)

X_ent, X_pru = Xm.loc[idx_ent], Xm.loc[idx_pru]
y_ent, y_pru = ym[idx_ent.to_numpy()], ym[idx_pru.to_numpy()]

for nombre, X_, y_ in [("Entrenamiento", X_ent, y_ent), ("Prueba", X_pru, y_pru)]:
    print(f"{nombre}: n={len(X_)} | desertores={y_['event'].sum()} ({y_['event'].mean():.1%})"
          f" | SP={X_['sede_sp'].mean():.1%} | 2019-21={X_['coh_2019_2021'].mean():.1%}"
          f" | 2022-23={X_['coh_2022_2023'].mean():.1%}"
          f" | tiempo {y_['time'].min():.0f}-{y_['time'].max():.0f}")

In [ ]:
import statsmodels
from statsmodels.duration.hazard_regression import PHReg

print("statsmodels:", statsmodels.__version__)
print("pandas:", pd.__version__)

In [ ]:
cox_sm = PHReg(endog=y_ent["time"], exog=X_ent,
               status=y_ent["event"].astype(int), ties="efron")
res_cox = cox_sm.fit()

coef = np.asarray(res_cox.params)
ic   = np.exp(np.asarray(res_cox.conf_int()))
tabla_cox = pd.DataFrame({
    "coef":     coef,
    "EE":       np.asarray(res_cox.bse),
    "HR":       np.exp(coef),
    "IC95_inf": ic[:, 0],
    "IC95_sup": ic[:, 1],
    "p":        np.asarray(res_cox.pvalues),
}, index=X_ent.columns).round(3)

print(tabla_cox)
print("\nn =", len(X_ent), "| eventos =", int(y_ent["event"].sum()))

In [ ]:
from scipy import stats

ev  = y_ent["event"]
r   = np.asarray(res_cox.schoenfeld_residuals)[ev]   # solo desertores
t   = y_ent["time"][ev]
d   = ev.sum()
V   = np.asarray(res_cox.cov_params())
r_esc = d * r @ V                                    # residuos escalados

g  = stats.rankdata(t)                               # tiempo en rangos
gc = g - g.mean()
chi2 = (gc @ r_esc) ** 2 / (d * np.diag(V) * (gc ** 2).sum())
p_ph = stats.chi2.sf(chi2, df=1)

print(pd.DataFrame({"chi2": chi2, "p": p_ph}, index=X_ent.columns).round(3))

In [ ]:
def prueba_ph(res, y, columnas):
    ev = y["event"]
    r  = np.asarray(res.schoenfeld_residuals)[ev]
    t  = y["time"][ev]
    d  = ev.sum()
    V  = np.asarray(res.cov_params())
    r_esc = d * r @ V
    gc = stats.rankdata(t)
    gc = gc - gc.mean()
    chi2 = (gc @ r_esc) ** 2 / (d * np.diag(V) * (gc ** 2).sum())
    return pd.DataFrame({"chi2": chi2, "p": stats.chi2.sf(chi2, df=1)},
                        index=columnas).round(3)

periodo_ent = periodo.loc[idx_ent].astype(str).to_numpy()
X_ent_s = X_ent.drop(columns=["coh_2019_2021", "coh_2022_2023"])

res_estr = PHReg(endog=y_ent["time"], exog=X_ent_s,
                 status=y_ent["event"].astype(int),
                 strata=periodo_ent, ties="efron").fit()

ic_e = np.exp(np.asarray(res_estr.conf_int()))
comp = pd.DataFrame({
    "HR_original":      tabla_cox.loc[X_ent_s.columns, "HR"].to_numpy(),
    "HR_estratificado": np.exp(np.asarray(res_estr.params)),
    "IC95_inf":         ic_e[:, 0],
    "IC95_sup":         ic_e[:, 1],
    "p":                np.asarray(res_estr.pvalues),
}, index=X_ent_s.columns).round(3)

print("Comparación de HR:\n", comp)
print("\nRiesgos proporcionales (modelo estratificado):\n",
      prueba_ph(res_estr, y_ent, X_ent_s.columns))

In [ ]:
import matplotlib.pyplot as plt
from statsmodels.nonparametric.smoothers_lowess import lowess

ev = y_ent["event"]
t  = y_ent["time"][ev]
r  = np.asarray(res_cox.schoenfeld_residuals)[ev]
V  = np.asarray(res_cox.cov_params())
beta_t = ev.sum() * r @ V + coef          # efecto estimado en cada tiempo

rng = np.random.default_rng(2026)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, var, titulo in zip(axes, ["sede_sp", "coh_2019_2021"],
                           ["Sede del interior vs. Sede Central", "Cohortes 2019-2021"]):
    j = list(X_ent.columns).index(var)
    ax.scatter(t + rng.uniform(-0.2, 0.2, len(t)), beta_t[:, j],
               s=6, alpha=0.3, color="grey")
    suav = lowess(beta_t[:, j], t, frac=0.4)
    ax.plot(suav[:, 0], suav[:, 1], color="black", lw=2, label="Tendencia suavizada")
    ax.axhline(coef[j], color="black", ls="--", lw=1, label="Efecto promedio (Cox)")
    ax.axhline(0, color="grey", ls=":", lw=1, label="Sin efecto")
    ax.set_title(titulo)
    ax.set_xlabel("Tiempo (cuatrimestres desde el fin del 1.er año)")
    ax.set_ylabel("Efecto estimado (log HR)")
    ax.set_xticks(range(1, 21, 2))
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.savefig("schoenfeld_sede_cohorte.png", dpi=300)
plt.show()

tramo = pd.cut(t, [0, 3, 6, 20], labels=["t1-3", "t4-6", "t7-20"])
for var in ["sede_sp", "coh_2019_2021"]:
    j = list(X_ent.columns).index(var)
    hr_tramo = np.exp(pd.Series(beta_t[:, j]).groupby(tramo, observed=True).mean())
    print(f"\n{var} – HR aproximado por tramo:")
    print(hr_tramo.round(2).to_string())
print("\nDesertores por tramo:", pd.Series(tramo).value_counts().sort_index().to_dict())

In [ ]:
cortes = [(0, 3, "t1_3"), (3, 6, "t4_6"), (6, 20, "t7_20")]
Xe = X_ent.reset_index(drop=True)

filas = []
for i in range(len(Xe)):
    T, E = y_ent["time"][i], y_ent["event"][i]
    for a, b, nom in cortes:
        if T > a:
            fila = Xe.iloc[i].to_dict()
            fila.update(entrada=a, salida=min(T, b),
                        evento=int(E and T <= b), tramo=nom)
            filas.append(fila)
ep = pd.DataFrame(filas)

for nom in ["t1_3", "t4_6", "t7_20"]:
    en_tramo = (ep["tramo"] == nom).astype(int)
    ep[f"sede_{nom}"]    = ep["sede_sp"] * en_tramo
    ep[f"coh1921_{nom}"] = ep["coh_2019_2021"] * en_tramo

print("Episodios:", len(ep), "| estudiantes:", len(Xe))
print("Eventos:", ep["evento"].sum(), "(esperado:", int(y_ent["event"].sum()), ")")
print(ep.groupby("tramo").agg(episodios=("evento", "size"), eventos=("evento", "sum")))
print("Tiempo-persona:", (ep["salida"] - ep["entrada"]).sum(), "vs", y_ent["time"].sum())
print(ep.head(4)[["entrada", "salida", "evento", "tramo",
                  "sede_sp", "sede_t1_3", "sede_t4_6", "sede_t7_20"]])

In [ ]:
vars_ext = ["materias_cursadas", "ausentes_x10", "materias_aprobadas",
            "coh_2022_2023", "ingreso_equiv",
            "sede_t1_3", "sede_t4_6", "sede_t7_20",
            "coh1921_t1_3", "coh1921_t4_6", "coh1921_t7_20"]

res_ext = PHReg(endog=ep["salida"], exog=ep[vars_ext], status=ep["evento"],
                entry=ep["entrada"], ties="efron").fit()

b   = np.asarray(res_ext.params)
V_e = np.asarray(res_ext.cov_params())
ic  = np.exp(np.asarray(res_ext.conf_int()))
tabla_ext = pd.DataFrame({"HR": np.exp(b), "IC95_inf": ic[:, 0],
                          "IC95_sup": ic[:, 1], "p": np.asarray(res_ext.pvalues)},
                         index=vars_ext).round(3)
print(tabla_ext)

def comparar(v1, v2):
    i, j = vars_ext.index(v1), vars_ext.index(v2)
    dif = b[i] - b[j]
    ee  = np.sqrt(V_e[i, i] + V_e[j, j] - 2 * V_e[i, j])
    p   = 2 * stats.norm.sf(abs(dif / ee))
    print(f"{v1} vs {v2}: cociente de HR = {np.exp(dif):.2f} "
          f"(IC95 {np.exp(dif - 1.96*ee):.2f}-{np.exp(dif + 1.96*ee):.2f}), p = {p:.3f}")

print()
for g in ["sede", "coh1921"]:
    comparar(f"{g}_t4_6", f"{g}_t1_3")
    comparar(f"{g}_t4_6", f"{g}_t7_20")
    

In [ ]:
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.metrics import (concordance_index_censored, concordance_index_ipcw,
                            integrated_brier_score)

# 1. Mismo Cox en scikit-survival
cox_sk = CoxPHSurvivalAnalysis(ties="efron").fit(X_ent, y_ent)
print("Coeficientes iguales a statsmodels:", np.allclose(cox_sk.coef_, coef, atol=1e-3))

# 2. Índice de concordancia en prueba
riesgo_cox = cox_sk.predict(X_pru)
c_harrell = concordance_index_censored(y_pru["event"], y_pru["time"], riesgo_cox)[0]
c_uno     = concordance_index_ipcw(y_ent, y_pru, riesgo_cox, tau=19)[0]

# 3. Integrated Brier Score (grilla t = 1 a 19)
tiempos  = np.arange(1, 20)
surv_cox = np.vstack([f(tiempos) for f in cox_sk.predict_survival_function(X_pru)])
ibs_cox  = integrated_brier_score(y_ent, y_pru, surv_cox, tiempos)

#    Referencia sin variables: Kaplan-Meier de entrenamiento, igual para todos
t_km, s_km = kaplan_meier_estimator(y_ent["event"], y_ent["time"])
pos     = np.searchsorted(t_km, tiempos, side="right") - 1
s_km_t  = np.where(pos >= 0, s_km[pos], 1.0)
ibs_km  = integrated_brier_score(y_ent, y_pru, np.tile(s_km_t, (len(y_pru), 1)), tiempos)

# 4. Guardar
metricas = pd.DataFrame({
    "C_Harrell": [0.5, c_harrell],
    "C_Uno":     [0.5, c_uno],
    "IBS":       [ibs_km, ibs_cox],
}, index=["Kaplan-Meier (sin variables)", "Cox"]).round(3)
print(metricas)

In [ ]:
from sksurv.ensemble import RandomSurvivalForest
import itertools

combos = list(itertools.product([5, 15, 30], ["sqrt", None]))
res_oob = []
for hoja, mf in combos:
    rsf = RandomSurvivalForest(n_estimators=500, min_samples_leaf=hoja,
                               max_features=mf, oob_score=True,
                               random_state=2026, n_jobs=-1)
    rsf.fit(X_ent, y_ent)
    res_oob.append({"min_samples_leaf": hoja,
                    "max_features": "todas" if mf is None else mf,
                    "C_OOB": rsf.oob_score_})

oob = pd.DataFrame(res_oob).sort_values("C_OOB", ascending=False).round(3)
print(oob.to_string(index=False))

In [ ]:
rsf_final = RandomSurvivalForest(n_estimators=500, min_samples_leaf=30,
                                 max_features="sqrt", oob_score=True,
                                 random_state=2026, n_jobs=-1).fit(X_ent, y_ent)

riesgo_rsf = rsf_final.predict(X_pru)
c_h_rsf = concordance_index_censored(y_pru["event"], y_pru["time"], riesgo_rsf)[0]
c_u_rsf = concordance_index_ipcw(y_ent, y_pru, riesgo_rsf, tau=19)[0]

surv_rsf = np.vstack([f(tiempos) for f in rsf_final.predict_survival_function(X_pru)])
ibs_rsf  = integrated_brier_score(y_ent, y_pru, surv_rsf, tiempos)

metricas.loc["RSF"] = [round(c_h_rsf, 3), round(c_u_rsf, 3), round(ibs_rsf, 3)]
print(metricas)

In [ ]:
rng = np.random.default_rng(2026)
n, B = len(y_pru), 1000
dif = []
for _ in range(B):
    i  = rng.integers(0, n, n)
    yb = y_pru[i]
    try:
        dc_h = (concordance_index_censored(yb["event"], yb["time"], riesgo_cox[i])[0]
                - concordance_index_censored(yb["event"], yb["time"], riesgo_rsf[i])[0])
        dc_u = (concordance_index_ipcw(y_ent, yb, riesgo_cox[i], tau=19)[0]
                - concordance_index_ipcw(y_ent, yb, riesgo_rsf[i], tau=19)[0])
        dibs = (integrated_brier_score(y_ent, yb, surv_cox[i], tiempos)
                - integrated_brier_score(y_ent, yb, surv_rsf[i], tiempos))
        dif.append((dc_h, dc_u, dibs))
    except ValueError:
        continue          # muestra sin casos suficientes en algún tiempo: se descarta

dif = np.array(dif)
print("Muestras válidas:", len(dif), "de", B)
for k, nombre in enumerate(["C Harrell", "C Uno", "IBS"]):
    d = dif[:, k]
    lo, hi = np.percentile(d, [2.5, 97.5])
    mejor = (d > 0).mean() if nombre != "IBS" else (d < 0).mean()
    print(f"{nombre} (Cox − RSF): {d.mean():+.3f}  IC95 [{lo:+.3f}, {hi:+.3f}]"
          f"  | Cox mejor en el {mejor:.0%} de las muestras")

In [ ]:
Xt = Xm.copy()
Xt["tasa_regularizacion"] = dm["tasa_regularizacion"]

rsf_t = RandomSurvivalForest(n_estimators=500, min_samples_leaf=30,
                             max_features="sqrt", random_state=2026,
                             n_jobs=-1).fit(Xt.loc[idx_ent], y_ent)
r_t = rsf_t.predict(Xt.loc[idx_pru])

print(f"RSF sin tasa: C Harrell {c_h_rsf:.3f} | C Uno {c_u_rsf:.3f}")
print(f"RSF con tasa: C Harrell "
      f"{concordance_index_censored(y_pru['event'], y_pru['time'], r_t)[0]:.3f} | C Uno "
      f"{concordance_index_ipcw(y_ent, y_pru, r_t, tau=19)[0]:.3f}")

In [ ]:
from sklearn.inspection import permutation_importance

imp = {}
for nombre, modelo in [("Cox", cox_sk), ("RSF", rsf_final)]:
    r = permutation_importance(modelo, X_pru, y_pru, n_repeats=30,
                               random_state=2026, n_jobs=1)
    imp[f"{nombre}_media"]  = r.importances_mean
    imp[f"{nombre}_desvío"] = r.importances_std

tabla_imp = (pd.DataFrame(imp, index=X_pru.columns)
               .sort_values("RSF_media", ascending=False).round(3))
print(tabla_imp)

In [ ]:
def imp_grupo(modelo, X, y, cols, n_rep=30, seed=2026):
    rng  = np.random.default_rng(seed)
    base = modelo.score(X, y)
    caidas = []
    for _ in range(n_rep):
        Xp   = X.copy()
        perm = rng.permutation(len(X))
        Xp[cols] = X[cols].to_numpy()[perm]
        caidas.append(base - modelo.score(Xp, y))
    return np.mean(caidas), np.std(caidas)

grupos = {
    "Desempeño 1.er año": ["materias_cursadas", "ausentes_x10", "materias_aprobadas"],
    "Sede":               ["sede_sp"],
    "Cohorte":            ["coh_2019_2021", "coh_2022_2023"],
    "Equivalencias":      ["ingreso_equiv"],
}

filas = []
for g, cols in grupos.items():
    fila = {"grupo": g}
    for nombre, modelo in [("Cox", cox_sk), ("RSF", rsf_final)]:
        m, s = imp_grupo(modelo, X_pru, y_pru, cols)
        fila[f"{nombre}_media"], fila[f"{nombre}_desvío"] = m, s
    filas.append(fila)

tabla_grupos = pd.DataFrame(filas).set_index("grupo").round(3)
print(tabla_grupos)

In [ ]:
import os, joblib
os.makedirs("resultados", exist_ok=True)

def guardar(df, nombre, indice=True):
    df.to_csv(f"resultados/{nombre}.csv", sep=";", decimal=",", index=indice)

guardar(tabla_cox,    "tabla_cox")
guardar(tabla_ext,    "tabla_cox_extendido")
guardar(metricas,     "metricas")
guardar(oob,          "rsf_oob", indice=False)
guardar(tabla_imp,    "importancia_individual")
guardar(tabla_grupos, "importancia_grupos")
guardar(pd.DataFrame(dif, columns=["dif_C_Harrell", "dif_C_Uno", "dif_IBS"]),
        "bootstrap_dif", indice=False)

joblib.dump({"X_ent": X_ent, "X_pru": X_pru, "y_ent": y_ent, "y_pru": y_pru,
             "idx_ent": idx_ent, "idx_pru": idx_pru, "tiempos": tiempos,
             "cox_sk": cox_sk, "rsf_final": rsf_final},
            "resultados/modelos_05.joblib")

print(sorted(os.listdir("resultados")))

In [ ]:
try:
    print(metricas)
    print("\nKernel vivo: rsf_final y cox_sk disponibles →",
          "rsf_final" in globals() and "cox_sk" in globals())
except NameError:
    print("Kernel reiniciado: hay que cargar modelos_05.joblib")

In [ ]:
import matplotlib.pyplot as plt

tg = tabla_grupos.loc[["Desempeño 1.er año", "Sede", "Cohorte", "Equivalencias"]]
etiquetas = ["Desempeño del\nprimer año", "Sede de ingreso", "Cohorte", "Ingreso por\nequivalencias"]
y = np.arange(len(tg))
h = 0.36

fig, ax = plt.subplots(figsize=(7, 3.8))
for mod, color, trama, desp in [("Cox", "#404040", None, -h/2),
                                ("RSF", "#8C8C8C", "//",  h/2)]:
    med, de = tg[f"{mod}_media"], tg[f"{mod}_desvío"]
    ax.barh(y + desp, med, height=h, color=color, hatch=trama,
            edgecolor="black", linewidth=0.5, label=mod,
            xerr=de, error_kw=dict(ecolor="black", lw=0.8, capsize=2))
    for yi, m, d in zip(y + desp, med, de):
        ax.text(m + d + 0.003, yi, f"{m:.3f}".replace(".", ","), va="center", fontsize=8)

ax.set_yticks(y, etiquetas)
ax.invert_yaxis()
ax.set_xlabel("Disminución del índice de concordancia (Harrell) al permutar el grupo")
ax.set_xlim(0, 0.16)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.2f}".replace(".", ",")))
ax.grid(axis="x", color="0.85", lw=0.5)
ax.set_axisbelow(True)
for lado in ["top", "right"]:
    ax.spines[lado].set_visible(False)
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
plt.savefig("importancia_grupos.png", dpi=300)
plt.show()

In [ ]:
from matplotlib.ticker import NullFormatter

# Valores exactos del Cox extendido
hr_ex = pd.Series(np.exp(b), index=vars_ext)
ic_ex = pd.DataFrame(np.exp(np.asarray(res_ext.conf_int())),
                     index=vars_ext, columns=["inf", "sup"])

tramos  = ["t1_3", "t4_6", "t7_20"]
nombres = {"t1_3":  "2.º año y 1.er cuatr. del 3.º\n(t1-3)",
           "t4_6":  "Fin del 3.er año y 4.º año\n(t4-6)",
           "t7_20": "5.º año en adelante\n(t7-20)"}
paneles = {"Sede del interior vs. Sede Central": "sede",
           "Cohortes 2019-2021 vs. 2015-2018": "coh1921"}

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for ax, (titulo, pref) in zip(axes, paneles.items()):
    for yi, tramo_k in enumerate(tramos):          # antes se llamaba "tr" y pisaba la tabla
        v = f"{pref}_{tramo_k}"
        if pref == "coh1921" and tramo_k == "t7_20":
            ax.text(2.1, yi, "no se muestra\n(censura cercana al corte)", ha="center",
                    va="center", fontsize=8, style="italic", color="0.4")
            continue
        hr, lo, hi = hr_ex[v], ic_ex.loc[v, "inf"], ic_ex.loc[v, "sup"]
        ax.errorbar(hr, yi, xerr=[[hr - lo], [hi - hr]], fmt="s", color="black",
                    ms=6, capsize=3, lw=1)
        ax.text(hr, yi - 0.22, f"{hr:.2f}".replace(".", ","), ha="center", fontsize=8)
    ax.axvline(1, color="0.5", ls="--", lw=0.8)
    ax.set_xscale("log")
    ax.set_xlim(0.5, 4.5)
    ax.set_xticks([0.5, 1, 2, 4], ["0,5", "1", "2", "4"])
    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.set_title(titulo, fontsize=10)
    ax.set_xlabel("Razón de riesgos (HR), escala logarítmica")
    for lado in ["top", "right"]:
        ax.spines[lado].set_visible(False)

axes[0].set_yticks(range(3), [nombres[t] for t in tramos])
axes[0].set_ylim(2.5, -0.6)
plt.tight_layout()
plt.savefig("hr_por_tramo.png", dpi=300)
plt.show()